# BLIP-1 Explorer — Pseudo-label Generation

**Teacher:** Fine-tuned BLIP-2 OPT-2.7B (`blip2_lora_uicd/merged_model`)  
**Dataset:** [kevintang2048/underwater-image-captioning-dataset-uicd](https://www.kaggle.com/datasets/kevintang2048/underwater-image-captioning-dataset-uicd)  
**Accelerator:** Kaggle Notebooks — T4 x2

---

## Purpose

Runs the fine-tuned BLIP-2 teacher over the full UICD to generate 5 diverse captions per image (1 beam-search + 4 sampled). Results are saved to `pseudo_labels.json` for use by `blip1-explorer-distill.ipynb`.

---

## Notebook Structure

| # | Section |
|---|--------|
| 1 | Environment Setup |
| 2 | GPU / Memory Check |
| 3 | Configuration |
| 4 | Dataset Loading |
| 5 | Train / Val / Test Split |
| 6 | Pseudo-label Generation |


## 1 · Environment Setup

In [ ]:
# ── Install packages not pre-installed in Kaggle Notebooks ──────────────────
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "peft>=0.5.0",
    "pycocoevalcap",
    "pycocotools",
    "open_clip_torch",
], check=True)

# ── Standard library ─────────────────────────────────────────────────────────
import gc
import json
import os
import random
import time
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ── HuggingFace ───────────────────────────────────────────────────────────────
from transformers import (
    AutoProcessor,
    Blip2ForConditionalGeneration,   # teacher (BLIP-2 OPT-2.7B)
    BlipForConditionalGeneration,    # student (BLIP-1 Large)
    BlipProcessor,
    get_linear_schedule_with_warmup,
)

# ── PEFT / LoRA ───────────────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, TaskType

print("All imports successful.")

## 2 · GPU / Memory Check

In [ ]:
gc.collect()
torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise EnvironmentError("No CUDA device found — enable the T4 x2 accelerator in Kaggle settings.")

device = torch.device("cuda")
n_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    total_vram = props.total_memory / 1024**3
    free_vram  = (props.total_memory - torch.cuda.memory_allocated(i)) / 1024**3
    print(f"  GPU {i}: {props.name}  |  Total VRAM: {total_vram:.1f} GB  |  Free VRAM: {free_vram:.1f} GB")

## 3 · Configuration

In [ ]:
CONFIG = {
    # ── Student model ─────────────────────────────────────────────────────────
    "student_model_id": "Salesforce/blip-image-captioning-large",

    # ── Teacher (fine-tuned BLIP-2 merged model) ──────────────────────────────
    "teacher_path": "/kaggle/input/models/kevintang2048/blip2-explorer/pytorch/default/2/blip2_lora_uicd/merged_model",
    # Processor subfolder exists but image preprocessor config is absent;
    # the cell falls back to the original model ID when loading fails.
    "teacher_processor_path": "/kaggle/input/models/kevintang2048/blip2-explorer/pytorch/default/2/blip2_lora_uicd/processor",

    # ── Dataset ──────────────────────────────────────────────────────────────
    "dataset_root": "/kaggle/input/datasets/kevintang2048/underwater-image-captioning-dataset-uicd",
    "split": (0.8, 0.1, 0.1),   # train / val / test fractions
    "seed": 42,

    # ── Pseudo-label generation ──────────────────────────────────────────────
    # 1 beam-search caption + (n_pseudo_captions - 1) sampled captions per image
    "n_pseudo_captions": 5,
    "pseudo_beam_width": 5,
    "pseudo_temperature": 0.7,
    "pseudo_top_p": 0.9,
    "pseudo_max_new_tokens": 40,
    "pseudo_label_path": "/kaggle/working/blip1_student_uicd/pseudo_labels.json",

    # ── Training ─────────────────────────────────────────────────────────────
    "epochs": 5,
    "batch_size": 4,             # slightly larger than teacher — student is smaller
    "lr": 2e-4,
    "max_length": 256,
    "warmup_ratio": 0.1,

    # ── CLIP auxiliary objective ─────────────────────────────────────────────
    # Raised from 0.2 (teacher) — student needs stronger CLIP alignment
    "clip_aux_weight": 0.35,
    "clip_aux_warmup_epochs": 1,
    "clip_aux_every_n_steps": 5,

    # ── Metric-aware validation policy ───────────────────────────────────────
    "full_metrics_every_n_epochs": 1,
    "metrics_subset_size": 128,
    "max_new_tokens_eval": 40,
    "num_beams_eval": 3,

    # Composite score weights — same as teacher for fair comparison
    "metric_weights": {
        "clip":   0.50,
        "cider":  0.25,
        "meteor": 0.10,
        "bleu4":  0.08,
        "rouge":  0.07,
    },
    "metric_norm": {
        "clip":   1.0,
        "cider":  2.0,
        "meteor": 1.0,
        "bleu4":  1.0,
        "rouge":  1.0,
    },

    # ── LoRA ─────────────────────────────────────────────────────────────────
    # Higher rank than teacher (16) to compensate for smaller model capacity
    "lora_r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "lora_bias": "none",
    # BLIP-1 uses a BERT-based text decoder whose attention Linear layers are
    # named "query", "key", "value" (not "q_proj"/"k_proj"/"v_proj" like OPT).
    # Verified via named_modules() scan in Section 9 before applying LoRA.
    "lora_target_modules": ["query", "key", "value"],

    # ── Output ───────────────────────────────────────────────────────────────
    "output_dir": "/kaggle/working/blip1_student_uicd",
}

# Create output directory (also parent of pseudo_label_path)
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

# Deterministic seeding
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


## 4 · Dataset Loading

Parses `UIC-captions.txt` (format: `<image_id>#<caption_index> <caption_text>`) and pairs each image filename with its list of reference captions. Identical to the teacher notebook.

In [ ]:
def load_captions(captions_path: Path) -> dict:
    """Parse UIC-captions.txt into {filename: [caption, ...]}."""
    captions_dict = {}
    with open(captions_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Format: <img_id>#<n> <caption text>
            parts = line.split(" ", 1)
            if len(parts) != 2:
                continue
            img_key, caption = parts
            filename = img_key.split("#")[0]
            captions_dict.setdefault(filename, []).append(caption.strip())
    return captions_dict


dataset_root = Path(CONFIG["dataset_root"])

captions_file = next(dataset_root.rglob("UIC-captions.txt"), None)
if captions_file is None:
    raise FileNotFoundError(f"UIC-captions.txt not found under {dataset_root}")

dataset_base = captions_file.parent
image_dir    = dataset_base / "uic_224x224_image"
if not image_dir.exists():
    image_dirs = [d for d in dataset_base.iterdir() if d.is_dir()]
    if not image_dirs:
        raise FileNotFoundError(f"No image directory found under {dataset_base}")
    image_dir = image_dirs[0]

captions_dict = load_captions(captions_file)

dataset = []
for filename, captions in captions_dict.items():
    img_path = image_dir / filename
    if img_path.exists():
        dataset.append({
            "image_path":     str(img_path),
            "image_filename": filename,
            "captions":       captions,
        })

print(f"Captions file  : {captions_file}")
print(f"Image directory: {image_dir}")
print(f"Total samples  : {len(dataset)}")
print(f"Sample entry   : {dataset[0]}")

## 5 · Train / Val / Test Split

Uses the same 80/10/10 split with `seed=42` as the teacher notebook so evaluation sets are directly comparable.

In [ ]:
train_frac, val_frac, test_frac = CONFIG["split"]

dataset_sorted = sorted(dataset, key=lambda x: x["image_filename"])
rng = random.Random(CONFIG["seed"])
rng.shuffle(dataset_sorted)

n = len(dataset_sorted)
n_train = int(n * train_frac)
n_val   = int(n * val_frac)

train_set = dataset_sorted[:n_train]
val_set   = dataset_sorted[n_train : n_train + n_val]
test_set  = dataset_sorted[n_train + n_val :]

print(f"Total : {n}")
print(f"Train : {len(train_set)}  ({len(train_set)/n*100:.1f}%)")
print(f"Val   : {len(val_set)}   ({len(val_set)/n*100:.1f}%)")
print(f"Test  : {len(test_set)}  ({len(test_set)/n*100:.1f}%)")

## 6 · Pseudo-label Generation (Phase 1)

Runs the fine-tuned BLIP-2 **teacher** over the **full UICD** (all three splits) to generate 5 diverse captions per image:
- 1 beam-search caption (`num_beams=5`) — high precision, good for CIDEr
- 4 sampled captions (`temperature=0.7`, `top_p=0.9`) — diversity for multi-reference coverage

Results are saved to `pseudo_labels.json`. The cell is **idempotent** — it is skipped if the file already exists, so re-running the notebook after a kernel restart is safe.

The teacher is fully unloaded and VRAM is freed before the student is loaded in Section 8.

In [ ]:
pseudo_label_path = Path(CONFIG["pseudo_label_path"])

# ── Skip if already generated ─────────────────────────────────────────────────
if pseudo_label_path.exists():
    with open(pseudo_label_path, "r") as f:
        pseudo_labels = json.load(f)
    print(f"Pseudo-labels already exist ({len(pseudo_labels)} entries). Skipping generation.")

else:
    # ── 1. Load teacher ───────────────────────────────────────────────────────
    print(f"Loading fine-tuned BLIP-2 teacher from: {CONFIG['teacher_path']}")

    # The saved processor folder may lack the image preprocessor config.
    # Try three locations in order and fall back to the HF hub if needed.
    _proc_candidates = [
        CONFIG["teacher_processor_path"],  # explicit processor subfolder
        CONFIG["teacher_path"],             # merged_model dir (sometimes has processor)
        "Salesforce/blip2-opt-2.7b",        # original HF model (always has it)
    ]
    teacher_processor = None
    for _src in _proc_candidates:
        try:
            teacher_processor = AutoProcessor.from_pretrained(_src)
            print(f"  Processor loaded from: {_src}")
            break
        except Exception as _e:
            print(f"  Processor not found at {_src}: {_e}")
    if teacher_processor is None:
        raise RuntimeError("Could not load BLIP-2 teacher processor from any candidate path.")

    # BLIP-2 OPT-2.7B is ~5.4 GB in fp16 — fits on one T4.
    # Using {"": 0} instead of "auto" avoids a multi-GPU device_map warning
    # where `language_model` is not listed as a top-level key by accelerate.
    teacher_model = Blip2ForConditionalGeneration.from_pretrained(
        CONFIG["teacher_path"],
        torch_dtype=torch.float16,
        device_map={"": 0},
    )
    teacher_model.eval()
    print("Teacher loaded on GPU 0.")

    # ── 2. Generate over the FULL dataset (train + val + test) ────────────────
    # Using all splits maximises the student's exposure to the domain.
    full_dataset = train_set + val_set + test_set
    n_sample = CONFIG["n_pseudo_captions"] - 1   # sampled count = total - 1 beam

    pseudo_labels = []
    for item in tqdm(full_dataset, desc="Generating pseudo-labels"):
        image = Image.open(item["image_path"]).convert("RGB")
        inputs = teacher_processor(images=image, return_tensors="pt").to("cuda:0")
        captions = []

        with torch.no_grad():
            # High-quality beam-search caption (anchors CIDEr consensus)
            beam_ids = teacher_model.generate(
                **inputs,
                num_beams=CONFIG["pseudo_beam_width"],
                max_new_tokens=CONFIG["pseudo_max_new_tokens"],
                no_repeat_ngram_size=3,
                repetition_penalty=1.2,
                early_stopping=True,
            )
            captions.append(
                teacher_processor.decode(beam_ids[0], skip_special_tokens=True).strip()
            )

            # Diverse sampled captions (expand multi-reference coverage)
            sample_ids = teacher_model.generate(
                **inputs,
                do_sample=True,
                temperature=CONFIG["pseudo_temperature"],
                top_p=CONFIG["pseudo_top_p"],
                max_new_tokens=CONFIG["pseudo_max_new_tokens"],
                num_return_sequences=n_sample,
                repetition_penalty=1.2,
            )
            for seq in sample_ids:
                captions.append(
                    teacher_processor.decode(seq, skip_special_tokens=True).strip()
                )

        pseudo_labels.append({
            "image_path":     item["image_path"],
            "image_filename": item["image_filename"],
            "captions":       captions,
        })

    # ── 3. Save ───────────────────────────────────────────────────────────────
    with open(pseudo_label_path, "w") as f:
        json.dump(pseudo_labels, f, indent=2)

    print(f"\nPseudo-labels saved: {len(pseudo_labels)} entries × {CONFIG['n_pseudo_captions']} captions")
    print(f"Path: {pseudo_label_path}")

    # ── Spot-check ────────────────────────────────────────────────────────────
    print("\n--- Spot-check (3 random samples) ---")
    for entry in random.sample(pseudo_labels, min(3, len(pseudo_labels))):
        print(f"\n  Image: {entry['image_filename']}")
        for i, cap in enumerate(entry["captions"]):
            tag = "beam" if i == 0 else f"sample {i}"
            print(f"    [{tag}] {cap}")

    # ── 4. Unload teacher — free VRAM before loading student ──────────────────
    del teacher_model
    del teacher_processor
    gc.collect()
    torch.cuda.empty_cache()
    print("Teacher unloaded. VRAM freed.")
